<a href="https://colab.research.google.com/github/juanpablor69/Proyecto_IA/blob/main/04%20-%20Modelo_con_preprocesado_clasico_y_Random_Forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Proyecto - Inteligencia Artificial para las Ciencias y las Ingenierías**
## Cuarto Notebook con uso de Random Forest.

## Accuracy: 0.4185559566787004. Score Kaggle: 0.41305
Autor: Juan Pablo Rendón Jimenez. \
Universidad de Antioquia

El **objetivo** de este notebook será aproximarnos a la solucion final usando distintos modelos junto a estrategias de preprocesado.






## Enlace con Kaggle

In [ ]:
import os
os.environ['KAGGLE_CONFIG_DIR'] = '.'
!chmod 600 ./kaggle.json
!kaggle competitions download -c udea-ai-4-eng-20252-pruebas-saber-pro-colombia

udea-ai-4-eng-20252-pruebas-saber-pro-colombia.zip: Skipping, found more recently modified local copy (use --force to force download)


## Lectura e inspeccion de datos

In [ ]:
!unzip udea*.zip > /dev/null
!wc *.csv

   296787    296787   4716673 submission_example.csv
   296787   4565553  59185238 test.csv
   692501  10666231 143732437 train.csv
  1286075  15528571 207634348 total


In [ ]:
import pandas as pd
import numpy as np

datos = pd.read_csv("train.csv")
print("Dimensiones del dataset:", datos.shape)

Dimensiones del dataset: (692500, 21)


La base de datos contiene 692500 filas y 21 columnas. La estructura es la siguiente:

In [ ]:
datos.head()

,ID,PERIODO_ACADEMICO,E_PRGM_ACADEMICO,E_PRGM_DEPARTAMENTO,E_VALORMATRICULAUNIVERSIDAD,E_HORASSEMANATRABAJA,F_ESTRATOVIVIENDA,F_TIENEINTERNET,F_EDUCACIONPADRE,F_TIENELAVADORA,...,E_PRIVADO_LIBERTAD,E_PAGOMATRICULAPROPIO,F_TIENECOMPUTADOR,F_TIENEINTERNET.1,F_EDUCACIONMADRE,RENDIMIENTO_GLOBAL,INDICADOR_1,INDICADOR_2,INDICADOR_3,INDICADOR_4
0,904256,20212,ENFERMERIA,BOGOTÁ,Entre 5.5 millones y menos de 7 millones,Menos de 10 horas,Estrato 3,Si,Técnica o tecnológica incompleta,Si,...,N,No,Si,Si,Postgrado,medio-alto,0.322,0.208,0.310,0.267
1,645256,20212,DERECHO,ATLANTICO,Entre 2.5 millones y menos de 4 millones,0,Estrato 3,No,Técnica o tecnológica completa,Si,...,N,No,Si,No,Técnica o tecnológica incompleta,bajo,0.311,0.215,0.292,0.264
2,308367,20203,MERCADEO Y PUBLICIDAD,BOGOTÁ,Entre 2.5 millones y menos de 4 millones,Más de 30 horas,Estrato 3,Si,Secundaria (Bachillerato) completa,Si,...,N,No,No,Si,Secundaria (Bachillerato) completa,bajo,0.297,0.214,0.305,0.264
3,470353,20195,ADMINISTRACION DE EMPRESAS,SANTANDER,Entre 4 millones y menos de 5.5 millones,0,Estrato 4,Si,No sabe,Si,...,N,No,Si,Si,Secundaria (Bachillerato) completa,alto,0.485,0.172,0.252,0.190
4,989032,20212,PSICOLOGIA,ANTIOQUIA,Entre 2.5 millones y menos de 4 millones,Entre 21 y 30 horas,Estrato 3,Si,Primaria completa,Si,...,N,No,Si,Si,Primaria completa,medio-bajo,0.316,0.232,0.285,0.294


## Analisis de la base de datos:

In [ ]:
print(datos.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 692500 entries, 0 to 692499
Data columns (total 21 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   ID                           692500 non-null  int64  
 1   PERIODO_ACADEMICO            692500 non-null  int64  
 2   E_PRGM_ACADEMICO             692500 non-null  object 
 3   E_PRGM_DEPARTAMENTO          692500 non-null  object 
 4   E_VALORMATRICULAUNIVERSIDAD  686213 non-null  object 
 5   E_HORASSEMANATRABAJA         661643 non-null  object 
 6   F_ESTRATOVIVIENDA            660363 non-null  object 
 7   F_TIENEINTERNET              665871 non-null  object 
 8   F_EDUCACIONPADRE             669322 non-null  object 
 9   F_TIENELAVADORA              652727 non-null  object 
 10  F_TIENEAUTOMOVIL             648877 non-null  object 
 11  E_PRIVADO_LIBERTAD           692500 non-null  object 
 12  E_PAGOMATRICULAPROPIO        686002 non-null  object 
 13 

In [ ]:
print("\n--- Valores faltantes por columna ---")
print(datos.isnull().sum())


--- Valores faltantes por columna ---
ID                                 0
PERIODO_ACADEMICO                  0
E_PRGM_ACADEMICO                   0
E_PRGM_DEPARTAMENTO                0
E_VALORMATRICULAUNIVERSIDAD     6287
E_HORASSEMANATRABAJA           30857
F_ESTRATOVIVIENDA              32137
F_TIENEINTERNET                26629
F_EDUCACIONPADRE               23178
F_TIENELAVADORA                39773
F_TIENEAUTOMOVIL               43623
E_PRIVADO_LIBERTAD                 0
E_PAGOMATRICULAPROPIO           6498
F_TIENECOMPUTADOR              38103
F_TIENEINTERNET.1              26629
F_EDUCACIONMADRE               23664
RENDIMIENTO_GLOBAL                 0
INDICADOR_1                        0
INDICADOR_2                        0
INDICADOR_3                        0
INDICADOR_4                        0
dtype: int64


## Limpieza de datos


In [ ]:
# 0. Eliminar columna duplicada
if 'F_TIENEINTERNET.1' in datos.columns:
    datos.drop(columns=['F_TIENEINTERNET.1'], inplace=True)

In [ ]:
# 1. Columnas categóricas (todas menos la objetivo)
columnas_categoricas = [
    'PERIODO_ACADEMICO',
    'E_PRGM_ACADEMICO',
    'E_PRGM_DEPARTAMENTO',
    'E_HORASSEMANATRABAJA',
    'F_ESTRATOVIVIENDA',
    'F_TIENEINTERNET',
    'F_EDUCACIONPADRE',
    'F_TIENELAVADORA',
    'F_TIENEAUTOMOVIL',
    'E_PRIVADO_LIBERTAD',
    'E_PAGOMATRICULAPROPIO',
    'F_TIENECOMPUTADOR',
    'F_EDUCACIONMADRE'
]

In [ ]:
# 2. Imputación categórica
for col in columnas_categoricas:
    datos[col] = datos[col].fillna("no info")

In [ ]:
# 3. Reemplazos especiales
datos['F_EDUCACIONMADRE'] = datos['F_EDUCACIONMADRE'].replace(
    ['No sabe', 'No Aplica'], 'no info'
)

# 4. Mapear columna ordinal
mapa_matricula = {
    'Menos de 500 mil': 0.25,
    'Entre 500 mil y menos de 1 millón': 0.75,
    'Entre 1 millón y menos de 2.5 millones': 1.75,
    'Entre 2.5 millones y menos de 4 millones': 3.25,
    'Entre 4 millones y menos de 5.5 millones': 4.75,
    'Entre 5.5 millones y menos de 7 millones': 6.25,
    'Más de 7 millones': 7.75,
    'No pagó matrícula': 0,
    'no info': -1
}
datos['E_VALORMATRICULAUNIVERSIDAD'] = datos['E_VALORMATRICULAUNIVERSIDAD'].map(mapa_matricula)

# 5. Imputación numérica
columnas_numericas = datos.select_dtypes(include=[np.number]).columns

--

In [ ]:
y = datos["RENDIMIENTO_GLOBAL"]
X = datos.drop(columns=["RENDIMIENTO_GLOBAL"])


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas),
        ("num", SimpleImputer(strategy="median"), columnas_numericas)
    ]
)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

In [ ]:
pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("rf", rf)
])

In [ ]:
pipe.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['PERIODO_ACADEMICO',
                                                   'E_PRGM_ACADEMICO',
                                                   'E_PRGM_DEPARTAMENTO',
                                                   'E_HORASSEMANATRABAJA',
                                                   'F_ESTRATOVIVIENDA',
                                                   'F_TIENEINTERNET',
                                                   'F_EDUCACIONPADRE',
                                                   'F_TIENELAVADORA',
                                                   'F_TIENEAUTOMOVIL',
                                                   'E_PRIVADO_LIBERTAD',
                                                   'E_PAGOMATRICULAPROPIO',
                                                   'F_TIENECOMPUTADOR',
                                                   'F_EDUCACIONMADRE']),
                                                 ('num',
                                                  SimpleImputer(strategy='median'),
                                                  Index(['ID', 'PERIODO_ACADEMICO', 'E_VALORMATRICULAUNIVERSIDAD', 'INDICADOR_1',
       'INDICADOR_2', 'INDICADOR_3', 'INDICADOR_4'],
      dtype='object'))])),
                ('rf',
                 RandomForestClassifier(class_weight='balanced',
                                        min_samples_leaf=2, min_samples_split=5,
                                        n_estimators=400, n_jobs=-1,
                                        random_state=42))])

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = pipe.predict(X_val)
acc = accuracy_score(y_val, y_pred)
acc


0.4185559566787004

In [ ]:
test = pd.read_csv("test.csv")
test['E_VALORMATRICULAUNIVERSIDAD'] = test['E_VALORMATRICULAUNIVERSIDAD'].map(mapa_matricula)
pred = pipe.predict(test)

In [ ]:
submission = pd.DataFrame({
    "ID": test["ID"],
    "RENDIMIENTO_GLOBAL": pred
})

submission.to_csv("submission.csv", index=False)

In [ ]:
!kaggle competitions submit -c udea-ai-4-eng-20252-pruebas-saber-pro-colombia -f submission.csv -m "Ramdom Forest N03 2do"

100% 3.97M/3.97M [00:00<00:00, 5.40MB/s]
Successfully submitted to UDEA/ai4eng 20252 - Pruebas Saber Pro Colombia